# 10 — Streaming RAG

Watch your LangGraph execute **node by node** in real time using `.stream(stream_mode="updates")`.

**What you'll learn**
- The difference between `.invoke()` (blocks until END) and `.stream()` (yields as each node finishes)
- `stream_mode="updates"` yields `{node_name: state_update}` — perfect for live debugging
- How to add a DuckDuckGo web-search fallback when local retrieval is sparse
- The `retrieve → web_fallback → generate` conditional pattern

**Companion examples:** 1-basic-local-rag (no streaming), 25-adaptive-rag (routing), 17-corrective-rag (doc grading)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langchain-chroma langchain-community chromadb duckduckgo-search python-dotenv langchain-text-splitters

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Knowledge base ──────────────────────────────────────────────────────────
from langchain_chroma import Chroma
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

SAMPLE_DOCS = [
    Document(
        page_content="Python lists are ordered, mutable sequences. Create with: my_list = [1, 2, 3]. Access by index: my_list[0] returns 1."
    ),
    Document(
        page_content="List methods: append(x) adds to end, extend(iterable) merges, pop(i) removes by index, sort() sorts in-place."
    ),
    Document(
        page_content="List comprehensions: squares = [x**2 for x in range(10)]. Filtered: evens = [x for x in range(10) if x % 2 == 0]."
    ),
    Document(
        page_content="Slicing: my_list[1:3] returns items at index 1 and 2. my_list[:2] takes the first two. my_list[::2] takes every other item."
    ),
    Document(
        page_content="Python tuples are immutable sequences: t = (1, 2, 3). Use tuples for fixed data; use lists when mutation is needed."
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(SAMPLE_DOCS)
vectorstore = Chroma.from_documents(chunks, OpenAIEmbeddings(), collection_name="streaming-rag")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
web_search = DuckDuckGoSearchResults(max_results=3)

print(f"Indexed {len(chunks)} chunks into ChromaDB")

In [ ]:
# ── 4. Build the streaming RAG graph ──────────────────────────────────────────
# Key concept: stream_mode="updates" yields {node_name: partial_state} as each
# node finishes. Unlike .invoke() which blocks until END, .stream() surfaces
# intermediate outputs the moment they're ready.
from typing import Literal, TypedDict

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph


class RAGState(TypedDict):
    question: str
    context: str
    answer: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using only the context below.\n\nContext:\n{context}"),
        ("human", "{question}"),
    ]
)


def retrieve(state: RAGState) -> dict:
    docs = retriever.invoke(state["question"])
    return {"context": "\n\n".join(d.page_content for d in docs)}


def web_fallback(state: RAGState) -> dict:
    results = web_search.invoke(state["question"])
    return {"context": str(results)}


def should_use_web(state: RAGState) -> Literal["web_fallback", "generate"]:
    # Fire the web fallback when local retrieval returns very little content
    return "web_fallback" if len(state["context"]) < 50 else "generate"


def generate(state: RAGState) -> dict:
    response = (prompt | llm).invoke({"context": state["context"], "question": state["question"]})
    return {"answer": response.content}


graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("web_fallback", web_fallback)
graph.add_node("generate", generate)
graph.add_edge(START, "retrieve")
graph.add_conditional_edges("retrieve", should_use_web)
graph.add_edge("web_fallback", "generate")
graph.add_edge("generate", END)
app = graph.compile()

print("Graph: START → retrieve → [web_fallback →] generate → END")

In [ ]:
# ── 5. Run with streaming — watch nodes fire one by one ───────────────────────
QUESTIONS = [
    "What is a Python list and how do you use it?",  # local vectorstore
    "Who won the most recent FIFA World Cup?",  # triggers web fallback
]

for question in QUESTIONS:
    print(f"\nQ: {question}")
    print("─" * 60)

    for chunk in app.stream(
        {"question": question, "context": "", "answer": ""},
        stream_mode="updates",  # ← yields {node_name: update} per node
    ):
        for node_name, update in chunk.items():
            print(f"[{node_name}]")
            if "context" in update and update["context"]:
                preview = update["context"][:100].replace("\n", " ")
                print(f"  context: {preview}...")
            if "answer" in update and update["answer"]:
                print(f"  answer: {update['answer']}")

## Exercises

**Exercise 1 — Change the fallback threshold:** The `should_use_web` gate fires when context is < 50 chars. Raise it to 200 and re-run. Which questions now hit the web path?

**Exercise 2 — Try `stream_mode="values"`:** Replace `stream_mode="updates"` with `stream_mode="values"`. How does the output change? (Hint: `values` streams the full accumulated state after each node, not just the delta.)

**Exercise 3 — Add a third node:** Insert a `grade` node between `retrieve` and `generate` that checks whether context is relevant to the question before generating.

## What's next

- **1-basic-local-rag** — same RAG pattern, no streaming
- **25-adaptive-rag** — classify queries before retrieval to pick the cheapest strategy
- **17-corrective-rag** — grade each retrieved doc, rewrite query if any score irrelevant